# MSS-Agent 1分钟入门

**世界上第一个内置'意义场自检'的开源 Agent 框架。**

## 本教程你会学到

1. **热税预算**: 如何让 Agent 拒绝无意义任务
2. **Δ检测**: 如何发现 Agent 陷入重复模式
3. **升维协议**: 多个 Agent 冲突如何升维解决

## 学习目标

学完后你能:
- 5行代码给任何 Agent 加热税检测
- 看懂 Δ 衰减曲线并触发蜕壳
- 在多 Agent 系统中用升维替代投票

In [ ]:
# 安装 (如果还没装)
!pip install mss-agent

## Step 1: 热税预算 — 拒绝无意义任务

![热税金字塔](docs/images/heat_tax_pyramid.png)

三层热税: L2意义(1000x) > L1逻辑(1x) > L0物理(0.001x)

In [ ]:
from mss_agent import HeatTaxBudget, HeatTaxLevel

# 创建热税预算
tax = HeatTaxBudget(threshold=2.0)

# 好任务: 意义热税极低
tax.charge(HeatTaxLevel.L2_MEANING, 0.002, "设计API架构")
print(f"设计API后: total={tax.total():.3f}, exceeded={tax.exceeded()}")

# 坏任务: 改写废话 → 高意义热税
tax2 = HeatTaxBudget(threshold=2.0)
tax2.charge(HeatTaxLevel.L2_MEANING, 0.06, "改写：你好")
print(f"改写后: total={tax2.total():.3f}, exceeded={tax2.exceeded()}")

## Step 2: Δ 检测 — Agent 是不是在重复?

![Δ衰减](docs/images/delta_decay.png)

Δ下降 → Agent陷入重复模式 → 触发蜕壳

In [ ]:
from mss_agent import DeltaProtocol, DeltaMemory

delta = DeltaProtocol(min_delta=0.3)
memory = DeltaMemory()

# 模拟: 重复相同任务4次
for i in range(6):
    task = "审查login代码" if i < 4 else "审查login代码的注入风险"
    novelty = memory.novelty_score(task)
    diversity = memory.diversity_score()
    d = delta.tick(f"task_{i}", novelty, diversity)
    memory.store(task, d)
    bar = '█' * int(d * 20)
    alert = ' ⚠️ MOLTING!' if delta.molting_alert else ''
    print(f"t{i}: Δ={d:.2f} {bar}{alert}")

print(f"\n健康: {delta.health()} | 蜕壳告警: {delta.molting_alert}")

## Step 3: MSS-Agent 完整示例

套在任何 LLM 外面:

In [ ]:
from mss_agent import MSSAgent

# 配置你的 LLM (这里用模拟)
def my_llm(prompt): return f"处理完成: {prompt[:30]}..."

agent = MSSAgent(name="MyAgent", llm=my_llm)

# 有意义任务 → 通过
r1 = agent.run("设计一个微服务的错误处理方案")
print(f"✅ {r1.output[:50]} | Δ={r1.delta:.2f}")

# 无意义任务 → 拦截
r2 = agent.run("")
print(f"🛑 拦截: {r2.aborted}, 原因: {r2.reason[:40]}")

# 健康报告
print(f"\n健康: {agent.health_report()['heat_tax']['total']:.2f}")

## Step 4: 升维 — 多Agent冲突怎么解决?

不是投票! 是找到双方被困在哪个维度, 加一维解决。

In [ ]:
from mss_agent.protocols import ElevationProtocol

ep = ElevationProtocol()
r = ep.resolve(
    "TravelAgent: 选直飞航班 (速度优先)",
    "CostAgent: 选最便宜航班 (成本优先)",
    "速度和成本冲突, 怎么选?"
)

print(f"被困维度: {r['trapped_dim']}")
print(f"升维到:   {r['elevation']}")
print(f"方案:     {r['resolution'][:100]}")
print("\n💡 没有选A也没有选B — 加了一个维度让冲突本身消失")

## 何时不该用 MSS-Agent?

- ❌ 你的 Agent 只是简单的 if-else (不需要 LLM)
- ❌ 你的任务100%确定性 (不需要意义场检测)
- ❌ 你的 Agent 从不出错也不重复 (那不需要 Δ)
- ❌ 你只有一个 Agent 且任务单一 (升维协议没用)

## 下一步

- 📖 [完整文档](https://mysama1.github.io/MSS-AI-Project/)
- 📦 [GitHub](https://github.com/mysama1/MSS-AI-Project)
- 🧪 [MAF 集成示例](examples/maf_integration_demo.py)
- 📊 [在线仪表盘](https://mysama1.github.io/MSS-AI-Project/dashboard/)